In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/delivery_data_features.csv")

print(df.shape)

(142502, 41)


In [3]:
df.columns.tolist()

['data',
 'trip_creation_time',
 'route_schedule_uuid',
 'route_type',
 'trip_uuid',
 'source_center',
 'source_name',
 'destination_center',
 'destination_name',
 'od_start_time',
 'od_end_time',
 'start_scan_to_end_scan',
 'is_cutoff',
 'cutoff_factor',
 'cutoff_timestamp',
 'actual_distance_to_destination',
 'actual_time',
 'osrm_time',
 'osrm_distance',
 'factor',
 'segment_actual_time',
 'segment_osrm_time',
 'segment_osrm_distance',
 'segment_factor',
 'trip_hour',
 'trip_dayofweek',
 'trip_month',
 'trip_weekend',
 'corridor',
 'corridor_volume',
 'source_hub_volume',
 'destination_hub_volume',
 'corridor_avg_actual_time',
 'corridor_avg_osrm_time',
 'corridor_avg_distance',
 'historical_corridor_delay',
 'historical_delay_ratio',
 'trip_segments',
 'trip_total_distance',
 'trip_total_actual_time',
 'trip_avg_speed']

In [4]:
target = "segment_actual_time"

drop_cols = [
    # IDs
    "trip_uuid",
    "route_schedule_uuid",

    # Target
    "segment_actual_time",

    # Strong leakage / future information
    "actual_time",
    "factor",
    "segment_factor",

    # Raw datetime columns
    "trip_creation_time",
    "od_start_time",
    "od_end_time",
    "cutoff_timestamp",

    # High-cardinality text columns
    "source_name",
    "destination_name",

    # String route identifier
    "corridor"
]

X = df.drop(columns=drop_cols)
y = df[target]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (142502, 28)
y Shape: (142502,)


In [5]:
X.dtypes.value_counts()

float64    14
int64       9
str         4
bool        1
Name: count, dtype: int64

In [6]:
X.select_dtypes(include=["object", "string", "str"]).columns.tolist()

['data', 'route_type', 'source_center', 'destination_center']

In [8]:
from sklearn.preprocessing import LabelEncoder

label_cols = [
    "data",
    "route_type",
    "source_center",
    "destination_center"
]

encoders = {}

for col in label_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

print("Encoding complete.")

Encoding complete.


In [9]:
X.dtypes.value_counts()

float64    14
int64      13
bool        1
Name: count, dtype: int64

In [10]:
"trip_uuid" in df.columns

True

In [11]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["trip_uuid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (114541, 28)
X_test : (27961, 28)
y_train: (114541,)
y_test : (27961,)


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [13]:
y_pred = rf.predict(X_test)

print("Predictions complete.")

Predictions complete.


In [15]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

MAE : 10.29
RMSE: 24.45
R²  : 0.7108


In [17]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(15)

,Feature,Importance
11,segment_osrm_distance,0.262464
23,historical_delay_ratio,0.136202
27,trip_avg_speed,0.121147
10,segment_osrm_time,0.101944
9,osrm_distance,0.046705
4,start_scan_to_end_scan,0.044436
7,actual_distance_to_destination,0.040343
26,trip_total_actual_time,0.038029
8,osrm_time,0.031754
6,cutoff_factor,0.031057


In [18]:
X_clean = X.drop(columns=[
    "trip_total_actual_time"
])

print(X_clean.shape)

(142502, 27)


In [19]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["trip_uuid"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_clean, y, groups=groups)
)

X_train = X_clean.iloc[train_idx]
X_test = X_clean.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [20]:
from sklearn.ensemble import RandomForestRegressor

rf_clean = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_clean.fit(X_train, y_train)

y_pred = rf_clean.predict(X_test)

In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

MAE : 10.33
RMSE: 24.58
R²  : 0.7078
